# Milk sum_losses Convergence Study: 2x2 Factorial Analysis

**Study overview:** Analyzes 400 training runs on the milk production dataset examining the
interaction between `active_g` and `sum_losses` in a 2x2 factorial design.

| Factor | Levels |
|--------|--------|
| `active_g` | False, True |
| `sum_losses` | False, True |

| Config | active_g | sum_losses | Runs |
|--------|----------|------------|------|
| `MilkSL_baseline` | False | False | 100 |
| `MilkSL_activeG` | True | False | 100 |
| `MilkSL_sumLosses` | False | True | 100 |
| `MilkSL_activeG+sumL` | True | True | 100 |

**Key questions:**
1. Does `sum_losses` help or hurt convergence on a small dataset?
2. How does `sum_losses` interact with `active_g`?
3. Are baseline/activeG results consistent with the main convergence study?

In [ ]:
import json
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.float_format", "{:.4f}".format)

%matplotlib inline

In [ ]:
# ── Helper functions ──

def parse_val_loss_curve(curve_str):
    """Parse val_loss_curve from CSV string to list of floats."""
    if pd.isna(curve_str):
        return []
    try:
        parsed = json.loads(curve_str)
        return [float(x) for x in parsed]
    except (json.JSONDecodeError, TypeError):
        try:
            parsed = ast.literal_eval(curve_str)
            return [float(x) for x in parsed]
        except Exception:
            return []

def cohens_d(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (group1.mean() - group2.mean()) / pooled_std

def effect_size_label(d):
    d = abs(d)
    if d < 0.2: return "negligible"
    if d < 0.5: return "small"
    if d < 0.8: return "medium"
    return "large"

def convergence_rate(df):
    return df["healthy"].sum() / len(df)

In [ ]:
# ── Load data ──

RESULTS_DIR = Path("../results")

df = pd.read_csv(RESULTS_DIR / "milk_sumloss_convergence" / "milk_sumloss_convergence_results.csv")
df["val_curve"] = df["val_loss_curve"].apply(parse_val_loss_curve)

# Ensure boolean columns
df["active_g"] = df["active_g"].astype(bool)
df["sum_losses"] = df["sum_losses"].astype(bool)

# Short labels
label_map = {
    "MilkSL_baseline": "baseline",
    "MilkSL_activeG": "activeG",
    "MilkSL_sumLosses": "sumLosses",
    "MilkSL_activeG+sumL": "activeG+sumL",
}
df["label"] = df["config_name"].map(label_map)

print(f"Loaded {len(df)} rows")
print(f"Configs: {df['config_name'].nunique()}")

## Data Inventory

In [ ]:
# Row counts and config distribution
print(f"Total rows: {len(df)}")
print(f"Parameters per model: {df['n_params'].iloc[0]:,}")
print(f"Stacks: {df['n_stacks'].iloc[0]}, Blocks/stack: {df['n_blocks_per_stack'].iloc[0]}")
print(f"Backcast: {df['backcast_length'].iloc[0]}, Forecast: {df['forecast_length'].iloc[0]}")
print()

summary = df.groupby("config_name").agg(
    runs=("run", "count"),
    active_g=("active_g", "first"),
    sum_losses=("sum_losses", "first"),
).reset_index()
print(summary.to_string(index=False))

## Convergence Rates

In [ ]:
# Bar chart: convergence rate by config
config_order = ["MilkSL_baseline", "MilkSL_activeG", "MilkSL_sumLosses", "MilkSL_activeG+sumL"]

fig, ax = plt.subplots(figsize=(8, 5))
conv_rates = []
labels = []
for cfg in config_order:
    sub = df[df["config_name"] == cfg]
    rate = convergence_rate(sub) * 100
    conv_rates.append(rate)
    labels.append(label_map[cfg])

colors = sns.color_palette("colorblind", 4)
bars = ax.bar(range(len(labels)), conv_rates, color=colors, edgecolor="black", linewidth=0.5)

for i, v in enumerate(conv_rates):
    ax.text(i, v + 1.5, f"{v:.0f}%", ha="center", fontweight="bold", fontsize=11)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Convergence Rate (%)")
ax.set_title("Convergence Rate by Configuration (2x2 Factorial)")
ax.set_ylim(0, 115)
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.3)

# Highlight sum_losses configs
for i in [2, 3]:
    bars[i].set_hatch("//")

plt.tight_layout()
plt.show()

## The sum_losses Catastrophe

`sum_losses` adds a weighted backcast reconstruction loss (0.25x) to the forecast loss. On this small
dataset, it appears to trap the optimizer in a suboptimal basin where it cannot converge to a good solution.

In [ ]:
# Summary stats for ALL runs (not just healthy) to show sum_losses behavior
print("Summary Statistics — ALL Runs (including non-converged)")
print("="*80)

all_stats = df.groupby("config_name")["best_val_loss"].agg(
    ["count", "mean", "std", "median", "min", "max"]
).round(4)
all_stats["CV%"] = (all_stats["std"] / all_stats["mean"] * 100).round(1)
all_stats["healthy"] = df.groupby("config_name")["healthy"].sum()
all_stats["conv_rate"] = (all_stats["healthy"] / all_stats["count"] * 100).round(1)

# Reorder
all_stats = all_stats.loc[[c for c in config_order if c in all_stats.index]]
print(all_stats.to_string())

In [ ]:
# Histogram: best_val_loss distribution for all configs
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, cfg in zip(axes.flat, config_order):
    sub = df[df["config_name"] == cfg]
    vals = sub["best_val_loss"].values
    lbl = label_map[cfg]
    
    ax.hist(vals, bins=30, edgecolor="black", alpha=0.7,
            color=sns.color_palette("colorblind")[config_order.index(cfg)])
    ax.set_title(f"{lbl} (n={len(vals)})")
    ax.set_xlabel("Best Val Loss (SMAPE)")
    ax.set_ylabel("Count")
    
    healthy = sub[sub["healthy"] == True]
    ax.text(0.95, 0.95,
            f"Healthy: {len(healthy)}/{len(sub)}\nMean: {vals.mean():.2f}\nStd: {vals.std():.2f}",
            transform=ax.transAxes, ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.8), fontsize=9)

plt.suptitle("Best Val Loss Distributions by Configuration", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Factorial Analysis

In [ ]:
# 2x2 heatmap: active_g x sum_losses -> mean convergence rate and mean loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Convergence rate heatmap
pivot_conv = df.groupby(["active_g", "sum_losses"])["healthy"].mean().unstack() * 100
sns.heatmap(pivot_conv, annot=True, fmt=".0f", cmap="RdYlGn", ax=axes[0],
            xticklabels=["sum_losses=F", "sum_losses=T"],
            yticklabels=["active_g=F", "active_g=T"],
            vmin=0, vmax=100, cbar_kws={"label": "%"})
axes[0].set_title("Convergence Rate (%)")

# Mean loss heatmap (all runs)
pivot_loss = df.groupby(["active_g", "sum_losses"])["best_val_loss"].mean().unstack()
sns.heatmap(pivot_loss, annot=True, fmt=".2f", cmap="RdYlGn_r", ax=axes[1],
            xticklabels=["sum_losses=F", "sum_losses=T"],
            yticklabels=["active_g=F", "active_g=T"],
            cbar_kws={"label": "SMAPE"})
axes[1].set_title("Mean Best Val Loss (All Runs)")

plt.suptitle("2x2 Factorial: active_g x sum_losses", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## activeG+sumL Attractor Basin

In [ ]:
# Show the tight cluster of activeG+sumL vs wider spread of sumLosses-only
sl_configs = ["MilkSL_sumLosses", "MilkSL_activeG+sumL"]

print("sum_losses Configurations — Detailed Statistics (All Runs)")
print("="*70)

for cfg in sl_configs:
    sub = df[df["config_name"] == cfg]
    vals = sub["best_val_loss"]
    print(f"\n{label_map[cfg]}:")
    print(f"  n={len(vals)}, mean={vals.mean():.4f}, std={vals.std():.4f}")
    print(f"  median={vals.median():.4f}, min={vals.min():.4f}, max={vals.max():.4f}")
    print(f"  CV% = {vals.std()/vals.mean()*100:.2f}%")
    print(f"  Healthy: {sub['healthy'].sum()}/{len(sub)}")

# Box plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
data = [df[df["config_name"] == cfg]["best_val_loss"].values for cfg in sl_configs]
bp = ax.boxplot(data, labels=[label_map[c] for c in sl_configs], patch_artist=True, widths=0.5)
for patch, color in zip(bp["boxes"], [sns.color_palette("colorblind")[2], sns.color_palette("colorblind")[3]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("Best Val Loss (SMAPE)")
ax.set_title("sum_losses Configs: Loss Distribution (All Runs)")

# Annotate spread
for i, cfg in enumerate(sl_configs):
    vals = df[df["config_name"] == cfg]["best_val_loss"]
    ax.text(i + 1, vals.max() + 1, f"std={vals.std():.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Statistical Tests

In [ ]:
# Mann-Whitney U: baseline vs activeG (non-sum_losses, healthy only)
print("="*70)
print("  Mann-Whitney U Tests (Healthy Runs)")
print("="*70)

base_h = df[(df["config_name"] == "MilkSL_baseline") & (df["healthy"] == True)]["best_val_loss"]
ag_h = df[(df["config_name"] == "MilkSL_activeG") & (df["healthy"] == True)]["best_val_loss"]

print(f"\n--- baseline vs activeG (non-sum_losses) ---")
print(f"  Baseline healthy: n={len(base_h)}, mean={base_h.mean():.4f}, median={base_h.median():.4f}")
print(f"  activeG  healthy: n={len(ag_h)}, mean={ag_h.mean():.4f}, median={ag_h.median():.4f}")

if len(base_h) >= 2 and len(ag_h) >= 2:
    U, p = stats.mannwhitneyu(base_h, ag_h, alternative="two-sided")
    d = cohens_d(base_h, ag_h)
    print(f"  Mann-Whitney U = {U:.1f}, p = {p:.6f} {'***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"  Cohen's d = {d:.4f} ({effect_size_label(d)})")

# Fisher exact for convergence rates
print(f"\n--- Fisher Exact: baseline vs activeG convergence rate ---")
base_all = df[df["config_name"] == "MilkSL_baseline"]
ag_all = df[df["config_name"] == "MilkSL_activeG"]
table = [
    [base_all["healthy"].sum(), len(base_all) - base_all["healthy"].sum()],
    [ag_all["healthy"].sum(), len(ag_all) - ag_all["healthy"].sum()],
]
odds, p = stats.fisher_exact(table)
print(f"  Baseline: {table[0][0]}/{sum(table[0])} healthy")
print(f"  activeG:  {table[1][0]}/{sum(table[1])} healthy")
print(f"  Odds ratio = {odds:.4f}, p = {p:.6f} {'***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")

# sum_losses configs comparison
print(f"\n--- Mann-Whitney U: sumLosses vs activeG+sumL (All Runs) ---")
sl_vals = df[df["config_name"] == "MilkSL_sumLosses"]["best_val_loss"]
agsl_vals = df[df["config_name"] == "MilkSL_activeG+sumL"]["best_val_loss"]
print(f"  sumLosses:    n={len(sl_vals)}, mean={sl_vals.mean():.4f}, std={sl_vals.std():.4f}")
print(f"  activeG+sumL: n={len(agsl_vals)}, mean={agsl_vals.mean():.4f}, std={agsl_vals.std():.4f}")
U, p = stats.mannwhitneyu(sl_vals, agsl_vals, alternative="two-sided")
d = cohens_d(sl_vals, agsl_vals)
print(f"  Mann-Whitney U = {U:.1f}, p = {p:.6f} {'***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
print(f"  Cohen's d = {d:.4f} ({effect_size_label(d)})")

## Validation Loss Curves

In [ ]:
# Plot val_loss_curve for all configs showing behavior differences
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, cfg in zip(axes.flat, config_order):
    sub = df[df["config_name"] == cfg]
    lbl = label_map[cfg]
    
    curves = sub["val_curve"].tolist()
    max_len = max(len(c) for c in curves if len(c) > 0)
    
    # Pad curves
    padded = np.full((len(curves), max_len), np.nan)
    for i, c in enumerate(curves):
        if len(c) > 0:
            padded[i, :len(c)] = c
    
    median = np.nanmedian(padded, axis=0)
    q25 = np.nanpercentile(padded, 25, axis=0)
    q75 = np.nanpercentile(padded, 75, axis=0)
    epochs = np.arange(max_len)
    
    color = sns.color_palette("colorblind")[config_order.index(cfg)]
    ax.plot(epochs, median, color=color, linewidth=1.5, label="Median")
    ax.fill_between(epochs, q25, q75, color=color, alpha=0.2, label="IQR")
    
    # Also plot a few individual curves (faint)
    n_sample = min(10, len(curves))
    rng = np.random.RandomState(42)
    idxs = rng.choice(len(curves), n_sample, replace=False)
    for idx in idxs:
        if len(curves[idx]) > 0:
            ax.plot(range(len(curves[idx])), curves[idx], color=color, alpha=0.1, linewidth=0.5)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Val Loss (SMAPE)")
    ax.set_title(f"{lbl} (n={len(sub)})")
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

plt.suptitle("Validation Loss Curves by Configuration", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Cross-Study Consistency

In [ ]:
# Compare baseline and activeG from this study vs Study 1
# Load Study 1 data for comparison
df_s1 = pd.read_csv(RESULTS_DIR / "milk_convergence" / "milk_convergence_results.csv")

print("="*70)
print("  Cross-Study Consistency: This Study vs 6-Stack Study")
print("="*70)

comparisons = [
    ("baseline", "MilkSL_baseline", "Milk6_baseline"),
    ("activeG", "MilkSL_activeG", "Milk6_activeG"),
]

for label, cfg_sl, cfg_s1 in comparisons:
    sl_h = df[(df["config_name"] == cfg_sl) & (df["healthy"] == True)]["best_val_loss"]
    s1_h = df_s1[(df_s1["config_name"] == cfg_s1) & (df_s1["healthy"] == True)]["best_val_loss"]
    
    print(f"\n--- {label} ---")
    print(f"  This study:  n={len(sl_h)}, mean={sl_h.mean():.4f}, median={sl_h.median():.4f}")
    print(f"  6-Stack:     n={len(s1_h)}, mean={s1_h.mean():.4f}, median={s1_h.median():.4f}")
    
    if len(sl_h) >= 2 and len(s1_h) >= 2:
        U, p = stats.mannwhitneyu(sl_h, s1_h, alternative="two-sided")
        print(f"  Mann-Whitney p = {p:.6f} {'***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns (consistent)'}")
    
    # Convergence rates
    rate_sl = convergence_rate(df[df["config_name"] == cfg_sl]) * 100
    rate_s1 = convergence_rate(df_s1[df_s1["config_name"] == cfg_s1]) * 100
    print(f"  Convergence: this={rate_sl:.0f}%, 6-stack={rate_s1:.0f}%")

## Key Findings

In [ ]:
# Auto-generated summary
print("="*70)
print("  KEY FINDINGS")
print("="*70)

# 1. sum_losses catastrophe
sl_rate = convergence_rate(df[df["config_name"] == "MilkSL_sumLosses"]) * 100
agsl_rate = convergence_rate(df[df["config_name"] == "MilkSL_activeG+sumL"]) * 100
print(f"\n1. sum_losses CATASTROPHE:")
print(f"   sumLosses convergence: {sl_rate:.0f}%")
print(f"   activeG+sumL convergence: {agsl_rate:.0f}%")
sl_mean = df[df["config_name"] == "MilkSL_sumLosses"]["best_val_loss"].mean()
agsl_mean = df[df["config_name"] == "MilkSL_activeG+sumL"]["best_val_loss"].mean()
print(f"   Both trapped at SMAPE ~{sl_mean:.1f} / {agsl_mean:.1f} (vs ~1.5-2 for healthy baselines)")

# 2. active_g consistency
base_rate = convergence_rate(df[df["config_name"] == "MilkSL_baseline"]) * 100
ag_rate = convergence_rate(df[df["config_name"] == "MilkSL_activeG"]) * 100
print(f"\n2. active_g BENEFIT (consistent with 6-stack study):")
print(f"   baseline: {base_rate:.0f}% convergence")
print(f"   activeG: {ag_rate:.0f}% convergence")

# 3. Factorial interaction
print(f"\n3. FACTORIAL INTERACTION:")
print(f"   active_g alone: improves convergence ({base_rate:.0f}% -> {ag_rate:.0f}%)")
print(f"   sum_losses alone: destroys convergence ({base_rate:.0f}% -> {sl_rate:.0f}%)")
print(f"   active_g + sum_losses: still catastrophic ({agsl_rate:.0f}%)")
print(f"   -> sum_losses dominates; active_g cannot rescue it")

# 4. Attractor basin
agsl_std = df[df["config_name"] == "MilkSL_activeG+sumL"]["best_val_loss"].std()
sl_std = df[df["config_name"] == "MilkSL_sumLosses"]["best_val_loss"].std()
print(f"\n4. ATTRACTOR BASIN TIGHTNESS:")
print(f"   activeG+sumL std: {agsl_std:.4f} (remarkably tight)")
print(f"   sumLosses std: {sl_std:.4f}")
print(f"   -> active_g narrows the basin even when stuck at a bad optimum")